# Assignment 2 — Exploratory Data Analysis (EDA)

## Industry problem statement 1
Customer analytics teams need to understand who is using products, where missing information exists, and which factors differ across segments before any model is trusted.

## Industry problem statement 2
Retail banking leadership wants evidence about customer outcomes by demographic/product segments without jumping straight to machine learning.

## Intuitive understanding
EDA is the process of interviewing the dataset before modelling: inspect what is there, ask questions, look for patterns, detect quality issues, and only then decide what modelling or business action makes sense.

## What are we trying to answer?
Which variables appear associated with survival, how complete is the dataset, and what patterns change across gender, class, age and embarkation?

## Dataset and important variables
`survived` = outcome; `pclass` = ticket class; `sex` = gender; `age` = age; `fare` = paid fare; `embarked` = embarkation point; `sibsp`/`parch` = family relationships.

**Dataset used for the primary Colab workflow:** Titanic dataset. The assignment explicitly names Titanic as its dataset source.

## What success means
Every basic, intermediate and advanced question is supported by a table or visual and followed by a plain-English interpretation.

> **Execution standard:** Each section follows the reference-notebook pattern: step heading → briefing → code → inspect the output → interpret it in business language.


## Step 1 — Import libraries

**Step briefing — What / Why / Expected output**

- **What we are doing:** Load the analysis and plotting toolkit.
- **Why it matters:** EDA needs inspection, statistics and visualisation.
- **What the output should tell us:** All required packages are available.

**Retail-banking lens:** Establish the analysis toolkit before viewing customer information.


In [ ]:
# Core data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Libraries loaded successfully.")


## Step 2 — Load and inspect the Titanic dataset

**Step briefing — What / Why / Expected output**

- **What we are doing:** Load the public Titanic dataset and inspect the first rows, dimensions and schema.
- **Why it matters:** EDA begins with understanding the data rather than assuming its structure.
- **What the output should tell us:** You should see the row count, columns and data types.

**Retail-banking lens:** This mirrors a banker checking a customer extract before analysis.


In [ ]:
try:
    df = sns.load_dataset("titanic").copy()
    DATA_SOURCE = "Seaborn Titanic (assignment dataset)"
except Exception:
    rng=np.random.default_rng(RANDOM_STATE)
    n=300
    df=pd.DataFrame({
        "survived":rng.integers(0,2,n),
        "pclass":rng.integers(1,4,n),
        "sex":rng.choice(["male","female"],n),
        "age":np.clip(rng.normal(32,14,n),1,80),
        "sibsp":rng.integers(0,4,n),
        "parch":rng.integers(0,4,n),
        "fare":np.clip(rng.lognormal(3.2,0.7,n),5,300),
        "embarked":rng.choice(["C","Q","S"],n),
    })
    DATA_SOURCE="Local smoke-test fallback (not official submission data)"
print("Data source:",DATA_SOURCE)
print("Shape:",df.shape)
display(df.head())
df.info()

## Step 3 — Missing values and data quality

**Step briefing — What / Why / Expected output**

- **What we are doing:** Calculate null counts, duplicate counts and data types.
- **Why it matters:** Missingness can bias conclusions and break later analysis.
- **What the output should tell us:** A data-quality profile for each field.

**Retail-banking lens:** A missing income or KYC field has different implications from a missing descriptive field.


In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean()*100).round(2),
    "unique": df.nunique(dropna=False)
}).sort_values("missing_pct", ascending=False)
display(quality)
print("Duplicate rows:", df.duplicated().sum())


## Step 4 — Basic analysis

**Step briefing — What / Why / Expected output**

- **What we are doing:** Calculate passenger counts, survival rate, age distribution and segment-level survival.
- **Why it matters:** These answer the assignment's basic questions directly.
- **What the output should tell us:** Core descriptive statistics and group comparisons.

**Retail-banking lens:** This is equivalent to customer count, outcome rate and segment performance reporting.


In [ ]:
print("Passenger count:", len(df))
print("Overall survival rate:", round(df["survived"].mean()*100,2), "%")
print("\nAge summary:")
display(df["age"].describe())
print("\nSurvival by gender:")
display(df.groupby("sex")["survived"].mean().mul(100).round(2))
print("\nSurvival by class:")
display(df.groupby("pclass")["survived"].mean().mul(100).round(2))


## Step 5 — Intermediate visual questions

**Step briefing — What / Why / Expected output**

- **What we are doing:** Examine survival by class and embarkation and relationships involving fare and age.
- **Why it matters:** Intermediate EDA should connect multiple variables instead of viewing them independently.
- **What the output should tell us:** Clear visual evidence for differences and relationships.

**Retail-banking lens:** A banking analyst would compare outcomes across channels, products and customer segments.


In [ ]:
fig, axes = plt.subplots(2,2, figsize=(14,10))
sns.barplot(data=df, x="sex", y="survived", errorbar=None, ax=axes[0,0]); axes[0,0].set_title("Survival Rate by Gender")
sns.barplot(data=df, x="pclass", y="survived", errorbar=None, ax=axes[0,1]); axes[0,1].set_title("Survival Rate by Class")
sns.barplot(data=df, x="embarked", y="survived", errorbar=None, ax=axes[1,0]); axes[1,0].set_title("Survival Rate by Embarkation")
sns.scatterplot(data=df, x="age", y="fare", hue="survived", alpha=.65, ax=axes[1,1]); axes[1,1].set_title("Age vs Fare")
plt.tight_layout(); plt.show()

numeric = df.select_dtypes(include=np.number)
plt.figure(figsize=(9,6))
sns.heatmap(numeric.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Numeric Correlation Matrix")
plt.show()


## Step 6 — Feature engineering

**Step briefing — What / Why / Expected output**

- **What we are doing:** Create family size, age group and title-like information from the available fields.
- **Why it matters:** Derived features can make patterns clearer and can later feed a model.
- **What the output should tell us:** New variables that represent business-friendly concepts.

**Retail-banking lens:** Banks routinely derive customer tenure bands, affordability ratios and household measures from raw fields.


In [ ]:
df_eda = df.copy()
df_eda["family_size"] = df_eda["sibsp"] + df_eda["parch"] + 1
df_eda["age_group"] = pd.cut(df_eda["age"], bins=[0,12,18,35,60,np.inf],
                             labels=["Child","Teen","Young Adult","Adult","Senior"])
display(df_eda.groupby("age_group", observed=True)["survived"].agg(["count","mean"]))


## Step 7 — Advanced multivariate analysis

**Step briefing — What / Why / Expected output**

- **What we are doing:** Use a pair plot and a combined segment table to explore several variables simultaneously.
- **Why it matters:** Multivariate EDA reveals interactions that one-variable charts can hide.
- **What the output should tell us:** A richer view of relationships across multiple variables.

**Retail-banking lens:** This is similar to asking how customer outcome changes when product, demographic and behaviour variables are considered together.


In [ ]:
sample_cols = ["survived","pclass","age","fare","family_size"]
sns.pairplot(df_eda[sample_cols].dropna().sample(min(400,len(df_eda)), random_state=RANDOM_STATE),
             hue="survived", corner=True)
plt.show()

segment_table = (
    df_eda.groupby(["sex","pclass"], observed=True)["survived"]
    .agg(["count","mean"])
    .rename(columns={"mean":"survival_rate"})
)
segment_table["survival_rate"] = segment_table["survival_rate"].mul(100).round(2)
display(segment_table)


## Step 8 — Preprocessing for optional modelling

**Step briefing — What / Why / Expected output**

- **What we are doing:** Demonstrate a clean analysis-ready dataset with selected fields and encoded categories.
- **Why it matters:** EDA should leave the data in a state that can transition into modelling.
- **What the output should tell us:** No unexpected missing values in selected modelling columns.

**Retail-banking lens:** This mirrors preparing a clean handoff from business analysis to credit/marketing modelling.


In [ ]:
model_df = df_eda[["survived","pclass","sex","age","fare","family_size","embarked"]].copy()
model_df["age"] = model_df["age"].fillna(model_df["age"].median())
model_df["embarked"] = model_df["embarked"].fillna(model_df["embarked"].mode()[0])
model_df = pd.get_dummies(model_df, columns=["sex","embarked"], drop_first=True)
print("Remaining missing values:", int(model_df.isna().sum().sum()))
display(model_df.head())


## Final Project Review

**What problem did I solve?**  
I performed a structured EDA of the Titanic dataset from data quality through basic, intermediate and advanced questions.

**What did the analysis/model learn?**  
EDA revealed group differences, missingness and relationships that would matter before any predictive model is built.

**Which result matters most?**  
The largest value comes from identifying which segments show meaningfully different outcomes and ensuring the data-quality issues are visible.

**What limitation must be disclosed?**  
Titanic is historical and not a modern retail-bank customer population; observed associations do not establish causation.

**What would I improve next?**  
Repeat the workflow on a current business dataset, test statistical significance, and build a predictive model only after the EDA findings are understood.
